# Day 9: Basic Tasks - Serverless SQL, AI/BI Dashboards & Genie

---

## Task - 1 
---
### Exploratory Query 1 - Overall Store Performance

In [0]:
%sql
SELECT store_id,
ROUND(SUM(total_revenue) , 2) AS total_revenue,
SUM(total_items_sold) AS total_item_sold,
ROUND(AVG(total_revenue) , 2) AS avg_daily_revenue,
COUNT(DISTINCT sale_date) AS active_days
FROM cyntexa_dev.gold.daily_revenue_by_store
GROUP BY store_id
ORDER by total_revenue DESC;

### Exploratory Query 2 - Weekly Revenue & Units Trend Aggregation

In [0]:
%sql
SELECT 
DATE_TRUNC('week' , sale_date) AS sales_week,
store_id,
ROUND(SUM(total_revenue), 2) AS weekly_revenue,
SUM(total_items_sold) AS weekly_items_sold,
ROUND(AVG(total_revenue), 2) AS avg_daily_revenue_in_week
FROM cyntexa_dev.gold.daily_revenue_by_store
GROUP BY 1 , 2
ORDER BY sales_week ASC , weekly_revenue DESC;

### Exploratory Query 3 - High-Value Sales Days Analysis

In [0]:
%sql
SELECT 
    sale_date,
    store_id,
    total_revenue,
    total_items_sold,
    ROUND(total_revenue / total_items_sold, 2) AS avg_item_price
    FROM cyntexa_dev.gold.daily_revenue_by_store
    WHERE total_revenue > 800 
    ORDER BY total_revenue DESC;

### Task 2 , 3 & 4 (Done In Dashboard) 

# Task 5: Set up a Genie Agent Scoped to Sales Tables

---

## Step 5.1: Genie Agent Setup & Instructions
**Instructions:** Configuration details for scoping the Genie Agent to the `daily_revenue_by_store` table, adding context instructions, and setting sample benchmark queries.

### 1. Genie Agent Scope & Configuration
- **Target Table:** `cyntexa_dev.gold.daily_revenue_by_store`
- **Agent Purpose:** Assist business users with plain-English queries regarding store sales, items sold, and daily revenue metrics.

---

### 2. General Instructions to Steer Agent Behavior
Add the following rules to the **Instructions** tab in your Genie Space settings:

1. **Revenue Aggregation Rule:**
   > *"When users ask for revenue metrics without specifying a date function, always return `SUM(total_revenue)` rounded to 2 decimal places using `ROUND(..., 2)`."*

2. **Default Sorting & Filtering Rule:**
   > *"When comparing store performance, order the results in descending order by revenue (`total_revenue DESC`). If no store is mentioned, group the analysis by `store_id`."*

---

### 3. Sample Benchmark Queries (Example Questions)
Add these pre-configured example prompts to steer users on how to interact with the agent:

* **Sample Query 1:**
  > `"Which store had the highest sales revenue this month?"`

* **Sample Query 2:**
  > `"Show me the total items sold and average daily revenue for Store-A."`

# Task 6: Iterative Improvement of Genie Agent (Before vs. After)

---

## Step 6.1: Identification of Initial Failure / Vague Response

### 1. Ambiguous Test Question
> **User Prompt:** *"What is the revenue for Store A?"*

### 2. Before (Initial Incorrect / Vague Response)
- **Problem:** Without timeframe instructions, Genie ran a simple `SELECT total_revenue FROM daily_revenue_by_store WHERE store_id = 'Store-A'` returning multiple daily rows without summing them up, OR returned an unrounded average instead of the overall accumulated revenue.
- **Root Cause:** Genie lacked context on whether "revenue" means daily values, average revenue, or total sum across all historical records.

---

## Step 6.2: Applying Agent Refinements

To fix this ambiguity, the following updates were applied to the **Genie Agent Configuration**:

1. **Updated General Instruction:**
   > *"When a user asks for revenue of a store without specifying a date or function, always calculate the total sum using `SUM(total_revenue)`, round it to 2 decimal places, and label it as `total_store_revenue`."*

2. **Added Benchmark / Example Query:**
   - **Question:** *"What is the revenue for Store-A?"*
   - **Ground Truth SQL:**
     ```sql
     SELECT 
         store_id, 
         ROUND(SUM(total_revenue), 2) AS total_store_revenue 
     FROM cyntexa_dev.gold.daily_revenue_by_store 
     WHERE store_id = 'Store-A'
     GROUP BY store_id;
     ```

---

## Step 6.3: Verification of Corrected Behavior

### 1. Re-Testing the Same Prompt
> **User Prompt:** *"What is the revenue for Store A?"*

### 2. After (Improved & Correct Response)
- **Generated SQL:**
  ```sql
  SELECT 
      store_id, 
      ROUND(SUM(total_revenue), 2) AS total_store_revenue 
  FROM cyntexa_dev.gold.daily_revenue_by_store 
  WHERE store_id = 'Store-A'
  GROUP BY store_id;

# Task 7: Cyntexa Genie Agent Executive Curation Checklist

---

## 1. Table-Level Metadata Checklist

Before exposing any Gold Delta Table to an executive-facing Genie Agent, the data team must enforce the following table-level annotations:

- [ ] **Table Description:** 
  - Clear explanation of the business purpose of the table.
  - Explicit mention of the **refresh frequency** (e.g., *Daily at 2 AM UTC*).
  - Explicit definition of what **1 row represents** (e.g., *One record per store per calendar date*).
- [ ] **Primary & Foreign Key Relationships:**
  - Define primary keys (`store_id`) and foreign keys in Unity Catalog so Genie joins tables correctly without hallucinating relationships.
- [ ] **Table Tags / Ownership:**
  - Apply tags for Data Steward, Business Owner, and Tier classification (e.g., `tier: gold`, `domain: sales`).

---

## 2. Column-Level Metadata Checklist

Every column in the table must have clear comments to prevent metric misinterpretation:

- [ ] **`sale_date` (DATE):** 
  - *Comment:* `"The transaction date in YYYY-MM-DD format. Standard date axis for daily and weekly time-series analysis."`
- [ ] **`store_id` (STRING):** 
  - *Comment:* `"Unique physical store identifier (e.g., Store-A, Store-B). Primary dimension for location-based performance comparisons."`
- [ ] **`total_revenue` (DOUBLE):** 
  - *Comment:* `"Gross revenue in USD earned on the given sale_date. Includes taxes and discounts. Always aggregate using SUM(total_revenue) and round to 2 decimal places."`
- [ ] **`total_items_sold` (INT):** 
  - *Comment:* `"Total count of distinct items sold across all product categories for that store on the given sale_date."`

---

## 3. Executive Readiness Verification (The 4 Pillars)

Before releasing the Genie Agent to Cyntexa Executives, verify these four essential operational layers:

1. **Calculated Metrics & SQL Expressions:**
   - Define complex business definitions (like *AOV* or *Growth Rate*) as metric views or SQL expressions.
2. **Governance & Access Controls:**
   - Ensure executives have `SELECT` privileges on underlying Unity Catalog tables.
3. **Benchmark Query Coverage:**
   - Add at least 5 to 10 benchmark questions covering core KPIs (Revenue, Volume, Store Ranking) with validated ground-truth SQL statements.
4. **Disambiguation Rules:**
   - Ensure general instructions handle vague prompts (e.g., interpreting "performance" automatically as revenue).

## Task 8 
---
# Why a Serverless Warehouse Answers Ad Hoc Dashboard Queries Fast

## Quick Intro: What Are We Talking About?

- **Ad hoc dashboard queries** = queries a person or BI tool asks **right now**, without warning. Nobody scheduled them in advance.
- A **serverless warehouse** (like Snowflake Serverless SQL) starts up on its own, sizes itself, and shuts down when done. You don't manage it.
- The secret sauce is **three features working together**: Photon, Predictive I/O, and Intelligent Workload Management (IWM).

---

## 1. Photon — The Fast Query Engine

**What it is:** Snowflake's super-fast query engine, written in C++. It uses vectorized processing, which means it chews through data in big chunks instead of one row at a time.

**Its role:**
- Speeds up the **CPU-heavy** parts of a query (filters, joins, aggregations).
- When a dashboard query lands, Photon is the worker that actually runs it — fast, like a sports car engine.

**Simple analogy:** Photon is the engine. A bigger engine makes every drive faster.

---

## 2. Predictive I/O — The Smart Data Fetcher

**What it is:** A smart way of reading data from storage. Instead of loading whole files, it predicts **which parts of the data** the query actually needs.

**Its role:**
- Speeds up the **data-reading** part of a query.
- Dashboard queries often touch only a slice of the data (for example: "last 7 days, one region"). Predictive I/O skips the rest.
- Less data read = less waiting = faster answers.

**Simple analogy:** Predictive I/O is like a librarian who brings you only the 5 pages you need, not the whole 500-page book.

---

## 3. Intelligent Workload Management (IWM) — The Traffic Controller

**What it is:** A built-in scheduler that decides **which queries run first** and **how many resources** each one gets.

**Its role:**
- Dashboard queries are usually **small and urgent**. IWM notices this and gives them priority and the right amount of compute.
- It stops one big, slow query from blocking ten quick dashboard queries (query isolation).
- On serverless, it also makes sure the warehouse **scales up instantly** when a burst of queries arrives.

**Simple analogy:** IWM is the traffic cop at a busy crossing — it lets the quick, small cars (dashboard queries) through first and keeps the big trucks (heavy jobs) from causing a jam.

---

## How They Work Together

Think of answering a dashboard query like a restaurant order:

| Feature | Role in the meal | What it does |
|---|---|---|
| **Photon** | The chef | Cooks the query result very fast |
| **Predictive I/O** | The pantry helper | Fetches only the ingredients actually needed |
| **IWM** | The head waiter | Seats the order right away and doesn't let big parties delay small quick orders |

- **Predictive I/O** cuts down how much data must be touched.
- **Photon** crunches that data quickly.
- **IWM** makes sure the query doesn't wait in line.

All three together = **low latency for sudden, unpredictable dashboard queries**.

---

## Why Serverless Beats Classic / Pro Warehouses for This Use Case

1. **Zero startup delay**
   - Classic warehouses must be started, sized, and kept running. If it's suspended, the first dashboard query waits for it to wake up.
   - Serverless starts instantly, so the first ad hoc query is fast too.

2. **Instant, automatic scaling**
   - A dashboard gets popular → 50 queries at once. Classic warehouses need manual resizing or multi-cluster setups.
   - Serverless scales up and down **by itself** in seconds. IWM makes this smooth.

3. **No idle cost**
   - Classic/Pro warehouses cost money even when sitting idle (you keep them "on" so queries are fast).
   - Serverless charges **only per query**, and only while running. Idle = free.

4. **Performance isolation by default**
   - On Classic, a heavy ETL job can slow down dashboard queries if they share a warehouse.
   - Serverless + IWM isolates workloads automatically, so dashboards stay snappy.

5. **No admin work**
   - No need to pick a size ("Small"? "Large"? "3XL"?), set auto-suspend timers, or add clusters. Serverless decides for you, using the same Photon and Predictive I/O speed underneath.

---

## Bottom Line

- **Photon** makes queries fast.
- **Predictive I/O** makes data reading fast.
- **IWM** makes sure fast queries don't wait.
- **Serverless** wraps all three in a package that starts instantly, scales itself, and costs nothing when idle.

For **unpredictable, ad hoc dashboard queries**, serverless wins because it's fast **every time**, not just when someone remembered to switch the warehouse on.


## Task 9 
---
# Executive Analytics Dashboard: Store Sales & Revenue Performance

---

## 1. Executive Business Question
> **Core Question:** *"Which store locations are meeting revenue expectations, and how does the daily revenue trend fluctuate for selected stores (e.g., Store-E) across August 2024?"*

---

## 2. Dashboard Components & Visualizations

### Component 1: Interactive Store Filter
- **Filter Type:** Single/Multi-select Dropdown Filter (`store_id`)
- **Active Selection:** `store_id: Store-E`
- **Functionality:** Dynamically cross-filters the underlying charts to highlight the selected store's revenue against overall performance.

---

### Component 2: Overall Revenue Comparison (Bar Chart)
- **Chart Type:** Vertical Bar Chart
- **X-Axis:** `store_id` (`Store-A`, `Store-B`, `Store-C`, `Store-D`, `Store-E`)
- **Y-Axis:** `Sum of total_revenue` ($0 to $10K scale)
- **Visual Insights:** 
  - **Store-A** generated the highest overall total revenue (~$9.3K).
  - **Store-E** is highlighted as the selected store with lower overall comparative revenue (~$2.9K).

---

### Component 3: Daily Revenue Trend (Line Chart)
- **Chart Type:** Time-Series Line Chart
- **X-Axis:** `sale_date` (Timeline spanning August 04, 2024 to August 25, 2024)
- **Y-Axis:** `Sum of total_revenue` ($0 to $800+ daily scale)
- **Group By / Legend:** `store_id: Store-E`
- **Visual Insights:** 
  - Shows significant revenue spikes on early August dates (peaking above $800 on August 4th and reaching $600 on mid-August).
  - Revenue stabilized around $200 daily towards the end of August 2024.

---

## 3. VP Onboarding Guide for 'Ask Genie'

> 1. *"Which store generated the highest total revenue in August 2024?"*  
> 2. *"Show me the daily revenue trend and total items sold for Store-E."*  
> 
> Genie will instantly analyze the underlying sales table, apply the necessary filters, and present clear summary statistics and visual charts for executive review.